<a href="https://colab.research.google.com/github/AlessandroCaforio/Python-for-ML-and-Econometrics/blob/main/Data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing

**[Let's deal with Missing data](https://)**

In [10]:
import pandas as pd
from io import StringIO

In [14]:
csv_data = \
'''A,B,C,D
 1.0,2.0,3.0,4.0
 5.0,6.0,,8.0
 10.0,11.0,12.0,'''

df = pd.read_csv(StringIO(csv_data))
df

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0
2,10.0,11.0,12.0,NaN


In [16]:
## Let's see what to use to understand how many missing values do we have (in each columsn)
print( df.isnull()) # Results: the same df with True (missing) False (not missing).
df.isnull().sum()

       A      B      C      D
0  False  False  False  False
1  False  False   True  False
2  False  False  False   True


,0
A,0
B,0
C,1
D,1


In [18]:
print(df.values)
print(df.value_counts)

[[ 1.  2.  3.  4.]
 [ 5.  6. nan  8.]
 [10. 11. 12. nan]]
<bound method DataFrame.value_counts of       A     B     C    D
0   1.0   2.0   3.0  4.0
1   5.0   6.0   NaN  8.0
2  10.0  11.0  12.0  NaN>


In [20]:
df.dropna(axis = 0) # here we are dropping every row with missing values.
df.dropna(axis = 1) # here we are dropping every column with missing values.
df.dropna(how = 'all') #it drops rows whre all columns are NaN (here it does not dop anything)
df.dropna(thresh = 4) #drop rows that have fewer than 4 real values
df.dropna(subset = ['C']) #Only drop rows where NaN appear in specific columns (here we putted C)

,A,B
0,1.0,2.0
1,5.0,6.0
2,10.0,11.0


To impute data, means to infere the missing values from the rest of the data that we have. The most common interpolation techniques is **mean imputation**, where we simply replace the missing value with the mean value of the entire feature column. The SimpleImputer class from sick-learn is perfect for this concept (it does not have sense to create a class from scratch since we already have everything computed in sklearn, although i usually don't make sense so i will try to build my class later).

In [23]:
from sklearn.impute import SimpleImputer
import numpy as np

imr = SimpleImputer(missing_values = np.nan, strategy = 'mean')
imr = imr.fit(df.values)
imputed_data = imr.transform(df.values)
imputed_data

array([[ 1. ,  2. ,  3. ,  4. ],
       [ 5. ,  6. ,  7.5,  8. ],
       [10. , 11. , 12. ,  6. ]])

Here we want to createa a simple version of the sklearn imputer, or more generally just any imputer.

The imputer meeds to work with both DataFrame and NumPy array.




In [36]:
class simpleimputer:

  def __init__(self, strategy : str = "mean"):
    '''
    Strategy is the only parameter: hot to compute the fill value (the interpolation)
    '''
    self.strategy = strategy
    self.statistics_ = None

  def fit(self, X):
    """
    Following sklearn, the fit is the object that compute the mean of each numeric column, ignoring NaNs.
    Than, it stores results in the self.statistics_, remembering that we want to have the mean for columns (one values for each columsn).
    This is direct implication of the concept of tidy data where each rows is an observation and each column a feature.
    """

    self._input_was_dataframe = isinstance(X, pd.DataFrame) ## Here the initial "_" means that is implemenation detail, nto meant to be used from outside.

    if self._input_was_dataframe:
      self._columns = X.columns
      self._index = X.index
      data = X

    else:
      data = pd.DataFrame(X) #If it's a numpy array, wrap it in a DataFrame temporarily.

    self.statistics_ = [] #it will be a list where (none) means "this column is non-numeric, no imputation needed"
    for col in data.columns:
      series = data[col] # This extract the columns of the DF as a pandas series
      has_missing = series.isna().any()

      if pd.api.types.is_numeric_dtype(series): # This returns True for int, float and other numeric dtypes
        fill_value = np.nanmean(series.values.astype(float)) #np.nanmean computes the mean ignoring missing values. The methods .values converts the sereis to numpy (as beofre)
        self.statistics_.append(fill_value)
      else:
        if has_missing:
          raise TypeError(
              f"Column '{col} is non-numeric and contains missing values "
              "Cannot impute with strategy = 'mean'."
          )
        self.statistics_.append(None)

    return self # THis is importat, so that later we can chain cells (like, imputer.fit(X).transform(X))


  def transform(self, X):
    if self.statistics_ is None:
      #This is a simple Guard, we need to follow the procedure, fit before transform, and this is a gently reminder.
      raise RuntimeError("fit() must be called before transform().")

    is_df = isinstance(X, pd.DataFrame)
    if is_df:
      out = X.copy() #This is critical, we never modify the original data.
      cols = list (X.columns)
    else:
      out = pd.DataFrame(X.copy())
      cols = list(out.columns)

    if len(cols) != len(self.statistics_):
      raise ValueError(
          f"X has {len(cols)} columns but fit() saw {len(self.statistics_)}."
      )

    for i, col in enumerate(cols): #enumerate() gives here both the index i and the column name co.
      fill_value = self.statistics_[i]
      if fill_value is not None: #Basically, we want to skip if the colmns is non numerical
        out[col] = out[col].fillna(fill_value)

    if is_df:
      return out
    else:
      return out.values

In [40]:
imp2 = simpleimputer(strategy = "mean")
resutls2 = imp2.fit(df)
imputed_data_2 = imp2.transform(df)
imputed_data_2

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,7.5,8.0
2,10.0,11.0,12.0,6.0


We've done it. Of course this is just an easy implementation of the methods, without the optimization of the sklearn classs (like errors handling and many ohter strategy). Please do not treat me as an idiot, but there is an even easier way to do so with a pandas methods.

In [41]:
df.fillna(df.mean())

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,7.5,8.0
2,10.0,11.0,12.0,6.0


Moving over, following the sklearn documentation we have different other methods of imputation like the one using the "Nearest neighbors imputaion" or KNNimputer

In [44]:
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors = 2, weights= "uniform")
imputer.fit_transform(df.values)

array([[ 1. ,  2. ,  3. ,  4. ],
       [ 5. ,  6. ,  7.5,  8. ],
       [10. , 11. , 12. ,  6. ]])

# Handling Categorical data
As of now we have only dealt with numerical data. In economics, econometrics and social sciences is very common to deal with categorical values.
An important dimension to keep in mind is the distinction between **ordinal** and **nominal** features.

Now, we'll use a slightly different Pandas dataframe. Toy one (always following the book).

In [46]:
df = pd.DataFrame([
  ['green', 'M', 10.1, 'class2'],
  ['red', 'L', 13.5, 'class1'],
  ['blue', 'XL', 15.3, 'class2']
  ])

df.columns = ['color', 'size', 'price', 'classlabel']
df


,color,size,price,classlabel
0,green,M,10.1,class2
1,red,L,13.5,class1
2,blue,XL,15.3,class2


This toy dataset helps us understand what is going on quite intuitively. The key idea is that some categorical variables are simply discretizations of an underlying continuous one. It is safe to say — and I mean this in the least body-shaming way possible — that clothing size represents an ordered scale ranging from small to large, with the same interpretability as a numeric scale from 1 to 5.

In [47]:
size_mapping = {'XL': 3,
                'L': 2,
                'M': 1}
#Now we can map it to the df:
df['size']=df['size'].map(size_mapping)
df

,color,size,price,classlabel
0,green,1,10.1,class2
1,red,2,13.5,class1
2,blue,3,15.3,class2


In [48]:
inv_size_mapping = {v: k for k, v in size_mapping.items()}
df['size'].map(inv_size_mapping)

,size
0,M
1,L
2,XL


## Encoding class labels
Now, we have dealt with ordinal features. We want to deal with nominal ones. The class columns is a perfect examples. Here being from calss1 or class2 does not have any meaning in terms of numerical relationship.
The next method will start the enumeration from zero, but will not have any numerical meaning.

In [52]:
class_mapping = {label: idx for idx, label in enumerate(np.unique(df['classlabel']))}

print(class_mapping)
df['classlabel'] = df['classlabel'].map(class_mapping)
print(df)

{np.int64(0): 0, np.int64(1): 1}
   color  size  price  classlabel
0  green     1   10.1           1
1    red     2   13.5           0
2   blue     3   15.3           1


In [53]:
from sklearn.preprocessing import LabelEncoder
class_le = LabelEncoder()
y = class_le.fit_transform(df['classlabel'].values)
y

array([1, 0, 1])

Before we simply used a dictionary mapping approach to convert ordinal size feature into integers.

In [54]:
X = df[['color', 'size', 'price']].values
color_le = LabelEncoder()
X[:, 0] = color_le.fit_transform(X[:, 0])
X

array([[1, 1, 10.1],
       [2, 2, 13.5],
       [0, 3, 15.3]], dtype=object)

Here there is an important issue. We transformed colors into ordinal values. But this is a mistake, if we feed this to our classifier, it will try to find a numerical relationship between the number associated to the color, whcih has no meaning, and the variable.

For this reason we will use the one-hot encoding. The idea here is to create dummy feature for each unique value in the nominal feature column.

In [58]:
from sklearn.preprocessing import OneHotEncoder

X = df[['color', 'size', 'price']].values
color_ohe = OneHotEncoder()
color_ohe.fit_transform(X[:, 0].reshape(-1,1)).toarray()


array([[0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.]])

In [61]:
from sklearn.compose import ColumnTransformer
X = df[['color', 'size', 'price']].values
c_transf = ColumnTransformer([
    ('onehot', OneHotEncoder(),[0]),
    ('nothing', 'passthrough', [1,2])
])

c_transf.fit_transform(X).astype(float)

array([[ 0. ,  1. ,  0. ,  1. , 10.1],
       [ 0. ,  0. ,  1. ,  2. , 13.5],
       [ 1. ,  0. ,  0. ,  3. , 15.3]])

In [64]:
df_wine = pd.read_csv(
'https://archive.ics.uci.edu/ml/'
'machine-learning-databases/wine/wine.data',
header=None
)

df_wine.columns = ['Class label', 'Alcohol',
 'Malic acid', 'Ash',
 'Alcalinity of ash', 'Magnesium',
 'Total phenols', 'Flavanoids',
 'Nonflavanoid phenols',
'Proanthocyanins',
 'Color intensity', 'Hue',
 'OD280/OD315 of diluted wines',
'Proline']
df_wine

,Class label,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,3,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,3,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,3,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,3,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


## Train and Test
This is a crucial features of any good ML pipeline. We want to split or partirion our dataset into a training and a testing one.
Here, sklearn helps us.

In [69]:
from sklearn.model_selection import train_test_split
X, y = df_wine.iloc[:, 1:].values, df_wine.iloc[:,0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0, stratify = y)

## Scaling features
Only **decision tree **and **random forest** are algorithms that do not need to worry about feature scaling, they are scale-invariant.

There are two commong approaches to bringin differnet features onto the same scale: **normalization** and **standardization**.

Oftenr, normalization refers to the rescaling of the features to a range of [0,1], which is a special case fo the **min-max scaling**.

$$
x^{(i)}_{norm} = \frac{x^{(i)}-x_{min}}{x_{max}-x_{min}}
$$

In [70]:
from sklearn.preprocessing import MinMaxScaler
mms = MinMaxScaler()
X_train_norm = mms.fit_transform(X_train)
X_test_norm = mms.fit_transform(X_test)

In [71]:
from sklearn.preprocessing import StandardScaler
stdc = StandardScaler()
X_train_std = stdc.fit_transform(X_train)
X_test_std = stdc.fit_transform(X_test)